# Fabric Defect Inspector: PatchCore (anomaly detection)

PatchCore learns what **normal fabric** looks like from defect-free crops only. At test time, any patch that looks unlike everything it has stored gets a high anomaly score. It needs no defect labels, so it can flag defects YOLO was never trained on.

This notebook is a compact PatchCore written directly in PyTorch (no extra library to install).

**Before you start**
1. On your PC, in a terminal:
   ```
   cd C:\Users\mehdi\Desktop\Fabric
   tar -a -c -f patchcore_textile.zip -C prepared\patchcore textile
   ```
2. Upload `patchcore_textile.zip` to the top level of your Google Drive (My Drive).
3. In Colab: Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.

## 1. Setup

In [ ]:
import torch, torchvision, numpy as np, pathlib, random, re, shutil, zipfile
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

assert torch.cuda.is_available(), "No GPU. Use Runtime > Change runtime type > T4 GPU, then run again."
dev = 'cuda'
print("GPU:", torch.cuda.get_device_name(0))
random.seed(0); np.random.seed(0); torch.manual_seed(0)

## 2. Get the crops from Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/patchcore_textile.zip'
import os
assert os.path.exists(ZIP_PATH), f"Not found: {ZIP_PATH}. Upload patchcore_textile.zip to My Drive or fix ZIP_PATH."

ROOT = pathlib.Path('/content/patchcore')
shutil.rmtree(ROOT, ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    for info in z.infolist():
        name = info.filename.replace('\\', '/')
        if name.endswith('/'):
            continue
        target = ROOT / name
        target.parent.mkdir(parents=True, exist_ok=True)
        with z.open(info) as src, open(target, 'wb') as dst:
            shutil.copyfileobj(src, dst)

def find(sub):
    hits = [p for p in ROOT.rglob('*') if p.is_dir() and p.as_posix().endswith(sub)]
    assert hits, f"folder '{sub}' not found in the zip"
    return sorted(hits[0].glob('*.png'))

train_good, test_good, test_defect = find('train/good'), find('test/good'), find('test/defect')
print(f"train/good {len(train_good)}   test/good {len(test_good)}   test/defect {len(test_defect)}")
print("expected:  train/good 1540   test/good 239   test/defect 195")

## 3. Feature extractor
A Wide ResNet-50 pretrained on ImageNet turns each 256 px crop into a 32 x 32 grid of patch features (layers 2 and 3 combined). The network is not trained here; it is only used to describe patches.

In [ ]:
SIZE = 256
weights = torchvision.models.Wide_ResNet50_2_Weights.IMAGENET1K_V1
net = torchvision.models.wide_resnet50_2(weights=weights).to(dev).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)

def load(paths):
    arr = np.stack([np.asarray(Image.open(p).convert('RGB').resize((SIZE, SIZE)), dtype=np.float32) / 255 for p in paths])
    x = torch.from_numpy(arr).permute(0, 3, 1, 2).to(dev)
    return (x - MEAN) / STD

@torch.no_grad()
def features(paths, bs=32):
    """Yields patch features for each batch: tensor (batch, patches, channels) on the GPU."""
    for i in range(0, len(paths), bs):
        x = load(paths[i:i + bs])
        x = net.maxpool(net.relu(net.bn1(net.conv1(x))))
        l2 = net.layer2(net.layer1(x))
        l3 = net.layer3(l2)
        f2 = F.avg_pool2d(l2, 3, 1, 1)
        f3 = F.interpolate(F.avg_pool2d(l3, 3, 1, 1), size=f2.shape[-2:], mode='bilinear', align_corners=False)
        f = torch.cat([f2, f3], 1)                                   # (B, 1536, 32, 32)
        yield f.permute(0, 2, 3, 1).reshape(f.shape[0], -1, f.shape[1])

f = next(features(train_good[:2]))
print("patch features per image:", tuple(f.shape[1:]), "(patches, channels)")

## 4. Build the memory bank
90% of the clean crops go into the memory bank. The other 10% are held back to set the alarm threshold in step 6.
From each image we keep a random sample of patches, which keeps the bank small enough to search quickly.

In [ ]:
PATCHES_PER_IMAGE = 40

paths = list(train_good); random.shuffle(paths)
n_cal = max(20, len(paths) // 10)
calib_paths, bank_paths = paths[:n_cal], paths[n_cal:]

g = torch.Generator().manual_seed(0)
parts = []
for f in features(bank_paths):
    B, P, C = f.shape
    idx = torch.stack([torch.randperm(P, generator=g)[:PATCHES_PER_IMAGE] for _ in range(B)]).to(dev)
    parts.append(torch.gather(f, 1, idx[..., None].expand(-1, -1, C)).reshape(-1, C))
bank = torch.cat(parts)
print(f"memory bank: {bank.shape[0]:,} patches x {bank.shape[1]} features from {len(bank_paths)} clean crops "
      f"({bank.numel() * 4 / 1e6:.0f} MB)")

## 5. Score images
For every patch in an image, find its nearest neighbour in the memory bank. The distance is that patch's anomaly score. The image score is its most anomalous patch.

In [ ]:
@torch.no_grad()
def score(paths, bs=16):
    scores, maps = [], []
    for f in features(paths, bs):
        for b in range(f.shape[0]):
            d = torch.cdist(f[b], bank).min(1).values               # (patches,)
            side = int(d.numel() ** 0.5)
            scores.append(d.max().item())
            maps.append(d.view(side, side).cpu())
    return np.array(scores), maps

cal_scores, _ = score(calib_paths)
good_scores, good_maps = score(test_good)
def_scores, def_maps = score(test_defect)
print(f"held-out clean: mean {cal_scores.mean():.2f}   test clean: mean {good_scores.mean():.2f}   test defect: mean {def_scores.mean():.2f}")

## 6. Results
**AUROC** measures how well the scores separate clean from defective crops: 1.0 is perfect, 0.5 is guessing.

The **threshold** is set from held-out clean crops only (95th percentile), without looking at any defect. So about 5% of clean crops are expected to trigger a false alarm.

In [ ]:
import pandas as pd

y = np.r_[np.zeros(len(good_scores)), np.ones(len(def_scores))]
s = np.r_[good_scores, def_scores]
auroc = roc_auc_score(y, s)
THRESHOLD = float(np.percentile(cal_scores, 95))

false_alarm = float((good_scores > THRESHOLD).mean())
detected = float((def_scores > THRESHOLD).mean())
print(f"Image AUROC: {auroc:.3f}")
print(f"Threshold: {THRESHOLD:.2f}")
print(f"Defect crops flagged: {detected:.1%}  ({int((def_scores > THRESHOLD).sum())} of {len(def_scores)})")
print(f"Clean crops flagged (false alarms): {false_alarm:.1%}  ({int((good_scores > THRESHOLD).sum())} of {len(good_scores)})")

rows = []
for cls in ('Hole', 'Knot', 'Line', 'Stain'):
    m = np.array([re.search(rf'_{cls}_\d+\.png$', p.name) is not None for p in test_defect])
    if m.any():
        rows.append({'class': cls, 'defect crops': int(m.sum()), 'flagged': int((def_scores[m] > THRESHOLD).sum()),
                     'detection rate': round(float((def_scores[m] > THRESHOLD).mean()), 3)})
by_class = pd.DataFrame(rows)
by_class

## 7. Score histogram
The further apart the two humps are, the better PatchCore separates clean from defective fabric.

In [ ]:
plt.figure(figsize=(8, 4))
bins = np.linspace(s.min(), s.max(), 40)
plt.hist(good_scores, bins, alpha=0.6, label=f'clean ({len(good_scores)})')
plt.hist(def_scores, bins, alpha=0.6, label=f'defect ({len(def_scores)})')
plt.axvline(THRESHOLD, color='k', ls='--', label=f'threshold {THRESHOLD:.2f}')
plt.xlabel('anomaly score'); plt.ylabel('crops'); plt.title(f'PatchCore on Textile crops (AUROC {auroc:.3f})'); plt.legend()
plt.savefig('/content/patchcore_scores.png', dpi=150, bbox_inches='tight'); plt.show()

## 8. Heatmaps
Top two rows: defect crops with the anomaly heatmap on top. Bottom row: clean crops. Each title shows the score and whether it crossed the threshold.

In [ ]:
import torchvision.transforms.functional as TF

def heat(m):
    m = F.interpolate(m[None, None], size=(SIZE, SIZE), mode='bilinear', align_corners=False)
    return TF.gaussian_blur(m, kernel_size=33, sigma=4.0)[0, 0].numpy()

vmax = max(float(np.percentile(def_scores, 95)), THRESHOLD * 1.2)
fig, ax = plt.subplots(3, 6, figsize=(18, 9.5))
picks = [(test_defect, def_maps, def_scores, np.linspace(0, len(test_defect) - 1, 12).astype(int)),
         (test_good, good_maps, good_scores, np.linspace(0, len(test_good) - 1, 6).astype(int))]
k = 0
for paths_, maps_, scores_, idx in picks:
    for i in idx:
        a = ax.flat[k]; k += 1
        a.imshow(Image.open(paths_[i]).convert('RGB').resize((SIZE, SIZE)))
        a.imshow(heat(maps_[i]), cmap='jet', alpha=0.4, vmin=0, vmax=vmax)
        flag = 'FLAGGED' if scores_[i] > THRESHOLD else 'ok'
        a.set_title(f"{'defect' if paths_ is test_defect else 'clean'}  {scores_[i]:.2f}  {flag}", fontsize=10)
        a.axis('off')
plt.tight_layout(); plt.savefig('/content/patchcore_heatmaps.png', dpi=130, bbox_inches='tight'); plt.show()

## 9. Save
Saves the memory bank, threshold and plots to the Drive folder `fabric_inspector_patchcore`.

In [ ]:
OUT = pathlib.Path('/content/drive/MyDrive/fabric_inspector_patchcore')
OUT.mkdir(exist_ok=True)
torch.save({'bank': bank.half().cpu(), 'threshold': THRESHOLD, 'size': SIZE, 'backbone': 'wide_resnet50_2',
            'layers': ['layer2', 'layer3'], 'auroc': float(auroc)}, OUT / 'patchcore_textile.pt')
pd.DataFrame([{'auroc': round(float(auroc), 3), 'threshold': round(THRESHOLD, 3), 'defects_flagged': round(detected, 3),
               'false_alarms': round(false_alarm, 3), 'bank_patches': int(bank.shape[0])}]).to_csv(OUT / 'patchcore_metrics.csv', index=False)
by_class.to_csv(OUT / 'patchcore_by_class.csv', index=False)
for f in ('/content/patchcore_scores.png', '/content/patchcore_heatmaps.png'):
    shutil.copy(f, OUT / pathlib.Path(f).name)
print("saved to", OUT, [p.name for p in OUT.iterdir()])